K.1 Menyiapkan Lingkungan dan Data

In [14]:
import os, sys, time, json, sqlite3, random, shutil, glob, subprocess
import numpy as np, pandas as pd

# Pasang pustaka (bila error ModuleNotFoundError: jalankan ulang sel ini, lalu Runtime -> Restart session)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "duckdb", "networkx", "shapely>=2.0", "tinydb", "deltalake>=1.0"], check=True)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/BigData"
else:
    BASE = os.path.abspath("./BigData")

DIR_P3 = f"{BASE}/Praktikum3"
DIR_SIMPAN = f"{BASE}/Praktikum4"
TRIPS_BERSIH = f"{DIR_P3}/trips_bersih"
os.makedirs(DIR_SIMPAN, exist_ok=True)
print("IN_COLAB:", IN_COLAB, "| TRIPS_BERSIH:", TRIPS_BERSIH, "| ada?", os.path.exists(TRIPS_BERSIH))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
IN_COLAB: True | TRIPS_BERSIH: /content/drive/MyDrive/BigData/Praktikum3/trips_bersih | ada? False


K.1b. Pembangkit data tiruan

In [15]:
def bangkitkan_trips_sintetis(keluar, n=1_000_000, seed=42):
    rng = np.random.default_rng(seed)
    wilayah = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island", "EWR"]
    zona_ids = np.arange(1, 264)
    wilayah_zona = rng.choice(wilayah, size=len(zona_ids), p=[0.35, 0.25, 0.22, 0.10, 0.05, 0.03])
    zona = pd.DataFrame({"LocationID": zona_ids, "borough_naik": wilayah_zona})
    pop = np.clip(rng.zipf(1.6, size=len(zona_ids)).astype(float), 1, 60)
    pop /= pop.sum()
    pu = rng.choice(zona_ids, size=n, p=pop)
    do = rng.choice(zona_ids, size=n, p=pop)
    awal = pd.Timestamp("2023-01-01")
    pickup = awal + pd.to_timedelta(rng.integers(0, 31 * 24 * 60, n), unit="m")
    durasi = rng.integers(2, 90, n)
    dist = np.round(rng.gamma(2.0, 1.6, n).clip(0.05, 60), 2)
    fare = np.round(2.5 + dist * 2.8 + rng.gamma(2.0, 1.0, n), 2)
    tip = np.round(rng.gamma(1.5, 1.2, n), 2)
    pay = rng.choice([1, 2, 3, 4], size=n, p=[0.7, 0.27, 0.02, 0.01])
    d = pd.DataFrame({
        "tpep_pickup_datetime": pickup,
        "tpep_dropoff_datetime": pickup + pd.to_timedelta(durasi, unit="m"),
        "passenger_count": rng.integers(1, 5, n).astype("float64"),
        "trip_distance": dist, "PULocationID": pu, "DOLocationID": do,
        "payment_type": pay, "fare_amount": fare, "tip_amount": tip,
        "total_amount": np.round(fare + tip + 3.0, 2),
        "durasi_menit": durasi.astype("float64"),
        "nama_pembayaran": pd.Series(pay).map({1: "Kartu kredit", 2: "Tunai", 3: "Gratis", 4: "Sengketa"}).values,
    })
    d = d.merge(zona.rename(columns={"LocationID": "PULocationID"}), on="PULocationID", how="left")
    d["suhu_c"] = np.round(rng.normal(3, 5, n), 1)
    d["hujan_mm"] = np.round(rng.gamma(0.4, 0.8, n), 2)
    if os.path.exists(keluar):
        shutil.rmtree(keluar)
    d.to_parquet(keluar, partition_cols=["borough_naik"], index=False)
    return keluar

DATA_TIRUAN = False
if not os.path.exists(TRIPS_BERSIH):
    print("!! trips_bersih/ tidak ditemukan -> memakai DATA TIRUAN.")
    TRIPS_BERSIH = bangkitkan_trips_sintetis(f"{DIR_SIMPAN}/trips_bersih_sintetis")
    DATA_TIRUAN = True
print("DATA_TIRUAN:", DATA_TIRUAN, "->", TRIPS_BERSIH)

!! trips_bersih/ tidak ditemukan -> memakai DATA TIRUAN.
DATA_TIRUAN: True -> /content/drive/MyDrive/BigData/Praktikum4/trips_bersih_sintetis


K.1 Pengukuran waktu, baca data, ganti nama kolom, buat contoh 300.000 baris

In [16]:
catatan = []
def ukur(label, fungsi):
    t0 = time.perf_counter()
    hasil = fungsi()
    detik = time.perf_counter() - t0
    catatan.append({"langkah": label, "detik": round(detik, 4)})
    print(f"[{label}] {detik:.4f} s")
    return hasil

def T(label):
    """Ambil waktu (detik) terakhir yang tercatat untuk sebuah label."""
    return [c["detik"] for c in catatan if c["langkah"] == label][-1]

HASIL = {}

df = ukur("baca trips_bersih", lambda: pd.read_parquet(TRIPS_BERSIH))

NAMA_BARU = {
    "tpep_pickup_datetime": "waktu_jemput",
    "PULocationID": "zona_jemput",
    "DOLocationID": "zona_tujuan",
    "borough_naik": "wilayah_jemput",
    "trip_distance": "jarak_mil",
    "total_amount": "total_bayar",
    "tip_amount": "tip",
    "nama_pembayaran": "metode_bayar",
}
df = df.rename(columns=NAMA_BARU)
df["wilayah_jemput"] = df["wilayah_jemput"].astype(str)
print("baris:", f"{len(df):,}", "| kolom:", df.shape[1])


sample = df.sample(n=min(300_000, len(df)), random_state=42).reset_index(drop=True)
sample["id_perjalanan"] = sample.index + 1
sample["waktu_jemput"] = sample["waktu_jemput"].astype(str)
HASIL["baris_df"], HASIL["kolom_df"] = len(df), df.shape[1]

[baca trips_bersih] 1.2823 s
baris: 1,000,000 | kolom: 15


K.2 Tabel biasa (SQLite) vs Key-Value

In [17]:
KOLOM = ["id_perjalanan", "waktu_jemput", "zona_jemput", "zona_tujuan",
         "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"]
data = sample[KOLOM].copy()

# Model 1 - relasional: tabel SQLite
if os.path.exists("uji.db"):
    os.remove("uji.db")
con = sqlite3.connect("uji.db")
ukur("relasional: muat tabel", lambda: data.to_sql("perjalanan", con, index=False))

# Model 2 - key-value: dict dengan kunci "perjalanan:<id>", nilai = teks JSON
def bangun_kv():
    return {f"perjalanan:{r['id_perjalanan']}": json.dumps(r) for r in data.to_dict("records")}
kv = ukur("key-value: bangun dict", bangun_kv)
print("jumlah key:", len(kv))

# Pertanyaan A: ambil SATU perjalanan berdasarkan id (200 id acak)
random.seed(42)
id_uji = random.sample(range(1, len(data) + 1), 200)
def cari_sql():
    cur = con.cursor()
    return [cur.execute("SELECT * FROM perjalanan WHERE id_perjalanan = ?", (i,)).fetchone() for i in id_uji]
def cari_kv():
    return [json.loads(kv[f"perjalanan:{i}"]) for i in id_uji]

r1 = ukur("cari 200 perjalanan - SQL tanpa indeks", cari_sql)
con.execute("CREATE INDEX idx_id ON perjalanan(id_perjalanan)")
r2 = ukur("cari 200 perjalanan - SQL dengan indeks", cari_sql)
r3 = ukur("cari 200 perjalanan - key-value", cari_kv)
print("hasil sama?", r1[0][0] == r2[0][0] == r3[0]["id_perjalanan"])

# Pertanyaan B: rata-rata total bayar per wilayah (melibatkan SEMUA data)
def hitung_sql():
    return con.execute("""SELECT wilayah_jemput, COUNT(*), ROUND(AVG(total_bayar), 2)
                          FROM perjalanan GROUP BY wilayah_jemput ORDER BY 2 DESC, 1""").fetchall()
def hitung_kv():
    total, jumlah = {}, {}
    for teks in kv.values():
        d = json.loads(teks)
        w = d["wilayah_jemput"]
        total[w] = total.get(w, 0) + d["total_bayar"]
        jumlah[w] = jumlah.get(w, 0) + 1
    return sorted([(w, jumlah[w], round(total[w] / jumlah[w], 2)) for w in jumlah],
                  key=lambda x: (-x[1], x[0]))

h1 = ukur("hitung per wilayah - SQL GROUP BY", hitung_sql)
h2 = ukur("hitung per wilayah - key-value (buka semua nilai)", hitung_kv)
print(h1[:3])
print("hasil sama?", h1 == h2)

[relasional: muat tabel] 3.5589 s
[key-value: bangun dict] 12.4842 s
jumlah key: 300000
[cari 200 perjalanan - SQL tanpa indeks] 5.3148 s
[cari 200 perjalanan - SQL dengan indeks] 0.0039 s
[cari 200 perjalanan - key-value] 0.0016 s
hasil sama? True
[hitung per wilayah - SQL GROUP BY] 0.2166 s
[hitung per wilayah - key-value (buka semua nilai)] 1.3225 s
[('Manhattan', 116930, 18.26), ('Queens', 81377, 18.29), ('Brooklyn', 62038, 18.24)]
hasil sama? True


K.3 Document: Bentuk Data yang Fleksibel

In [18]:
from tinydb import TinyDB, Query
from tinydb.storages import MemoryStorage

def ke_dokumen(r):
    dok = {
        "id_perjalanan": int(r.id_perjalanan),
        "waktu_jemput": r.waktu_jemput,
        "jemput": {"zona": int(r.zona_jemput), "wilayah": r.wilayah_jemput},
        "tujuan": {"zona": int(r.zona_tujuan)},
        "biaya": {"jarak_mil": float(r.jarak_mil),
                  "total": float(r.total_bayar),
                  "metode_bayar": r.metode_bayar},
    }
    if r.metode_bayar == "Kartu kredit":
        dok["biaya"]["tip"] = float(r.tip)
    return dok

db = TinyDB(storage=MemoryStorage)
tabel = db.table("perjalanan")
docs = [ke_dokumen(r) for r in sample.head(20_000).itertuples()]
tabel.insert_multiple(docs)
print("jumlah dokumen:", len(tabel))
print(json.dumps(tabel.all()[0], indent=2))

Q = Query()
print("jemput di Manhattan & total > 50 :", len(tabel.search((Q.jemput.wilayah == "Manhattan") & (Q.biaya.total > 50))))
n_tip = len(tabel.search(Q.biaya.tip.exists()))
n_no_tip = len(tabel.search(~Q.biaya.tip.exists()))
print("dokumen yang punya field tip :", n_tip)
print("dokumen yang TIDAK punya tip :", n_no_tip)

# Menambah field baru ("cuaca") pada satu dokumen - tanpa ALTER TABLE / migrasi
tabel.insert({"id_perjalanan": 999_999, "waktu_jemput": "2023-01-15 08:00:00",
              "jemput": {"zona": 161, "wilayah": "Manhattan"}, "tujuan": {"zona": 236},
              "biaya": {"jarak_mil": 1.2, "total": 14.5, "metode_bayar": "Kartu kredit", "tip": 3.0},
              "cuaca": {"hujan": True, "suhu_c": -2.0}})
n_cuaca = len(tabel.search(Q.cuaca.exists()))
print("dokumen yang punya field cuaca :", n_cuaca)
print("total dokumen sekarang :", len(tabel))

# Risiko skema fleksibel: salah ketik nama field TIDAK ditegur database
tabel.insert({"id_perjalanan": 888_888, "biya": {"total": 10.0}})   # "biya" bukan "biaya"
print("dokumen salah ketik diterima tanpa error ->", len(tabel.search(Q.biya.exists())), "dokumen")
tabel.remove(Q.id_perjalanan == 888_888)
HASIL.update(n_tip=n_tip, n_no_tip=n_no_tip, n_cuaca=n_cuaca)

jumlah dokumen: 20000
{
  "id_perjalanan": 1,
  "waktu_jemput": "2023-01-10 17:57:00",
  "jemput": {
    "zona": 219,
    "wilayah": "Staten Island"
  },
  "tujuan": {
    "zona": 46
  },
  "biaya": {
    "jarak_mil": 2.0,
    "total": 17.99,
    "metode_bayar": "Kartu kredit",
    "tip": 4.81
  }
}
jemput di Manhattan & total > 50 : 9
dokumen yang punya field tip : 13979
dokumen yang TIDAK punya tip : 6021
dokumen yang punya field cuaca : 1
total dokumen sekarang : 20001
dokumen salah ketik diterima tanpa error -> 1 dokumen


K.4 Penyimpanan per Kolom DuckDB pada Parquet

In [19]:
import duckdb
GLOB = f"{TRIPS_BERSIH}/*/*.parquet"
con_d = duckdb.connect()

# VIEW = "jendela" bernama di atas file Parquet (data tidak disalin); nama kolom TLC diganti di sini.
con_d.sql(f"""
    CREATE VIEW perjalanan AS
    SELECT tpep_pickup_datetime AS waktu_jemput,
           PULocationID AS zona_jemput,
           DOLocationID AS zona_tujuan,
           borough_naik AS wilayah_jemput,
           trip_distance AS jarak_mil,
           total_amount AS total_bayar,
           nama_pembayaran AS metode_bayar
    FROM read_parquet('{GLOB}', hive_partitioning=true)""")

SQL = """SELECT wilayah_jemput, COUNT(*) AS jumlah, ROUND(AVG(total_bayar), 2) AS rata_bayar
         FROM perjalanan GROUP BY wilayah_jemput ORDER BY jumlah DESC, wilayah_jemput"""
hasil_d = ukur("DuckDB - hitung per wilayah (langsung dari Parquet)", lambda: con_d.sql(SQL).df())
print(hasil_d)

# Pembanding: SQLite (per baris) pada data yang sama
KOLOM_S = ["wilayah_jemput", "total_bayar", "jarak_mil", "zona_jemput", "zona_tujuan"]
if os.path.exists("baris.db"):
    os.remove("baris.db")
con_s = sqlite3.connect("baris.db")
ukur("SQLite - muat seluruh baris ke tabel", lambda: df[KOLOM_S].to_sql("perjalanan", con_s, index=False))
hasil_s = ukur("SQLite - hitung per wilayah", lambda: con_s.execute(
    """SELECT wilayah_jemput, COUNT(*), ROUND(AVG(total_bayar), 2)
       FROM perjalanan GROUP BY wilayah_jemput ORDER BY 2 DESC, 1""").fetchall())
print("hasil sama?", [tuple(r) for r in hasil_d.itertuples(index=False)] == [tuple(r) for r in hasil_s])

# Filter pada kolom partisi -> partition pruning (Praktikum 3)
SQL2 = """SELECT COUNT(*), ROUND(AVG(total_bayar), 2) FROM perjalanan WHERE wilayah_jemput = 'Staten Island'"""
print(ukur("DuckDB - hanya wilayah Staten Island", lambda: con_d.sql(SQL2).fetchall()))

ukuran_pq = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(TRIPS_BERSIH) for f in fs) / 1024**2
ukuran_sq = os.path.getsize("baris.db") / 1024**2
print(f"Parquet ({df.shape[1]} kolom): {ukuran_pq:.1f} MB | SQLite ({len(KOLOM_S)} kolom): {ukuran_sq:.1f} MB")
HASIL.update(ukuran_pq=ukuran_pq, ukuran_sq=ukuran_sq)

[DuckDB - hitung per wilayah (langsung dari Parquet)] 0.6241 s
  wilayah_jemput  jumlah  rata_bayar
0      Manhattan  391107       18.26
1         Queens  270929       18.27
2       Brooklyn  205487       18.29
3  Staten Island   67041       18.28
4          Bronx   60912       18.21
5            EWR    4524       18.31
[SQLite - muat seluruh baris ke tabel] 4.2900 s
[SQLite - hitung per wilayah] 0.6314 s
hasil sama? True
[DuckDB - hanya wilayah Staten Island] 0.0455 s
[(67041, 18.28)]
Parquet (15 kolom): 32.2 MB | SQLite (5 kolom): 37.6 MB


K.5 Graph: Pertanyaan tentang Keterhubungan

In [20]:
import networkx as nx

arus = ukur("hitung arus antar zona", lambda: con_d.sql("""
    SELECT zona_jemput AS asal, zona_tujuan AS tujuan, COUNT(*) AS jumlah
    FROM perjalanan GROUP BY 1, 2""").df())

G = nx.DiGraph()
for r in arus.itertuples(index=False):
    G.add_edge(int(r.asal), int(r.tujuan), jumlah=int(r.jumlah))
print("node (zona):", G.number_of_nodes(), "| edge (arus):", G.number_of_edges())
HASIL.update(n_node=G.number_of_nodes(), n_edge=G.number_of_edges())

# Lima arus terbesar; pengurut tambahan (jumlah, asal, tujuan) agar urutan pasti (lihat M.5)
top_graf = sorted(G.edges(data=True), key=lambda e: (-e[2]["jumlah"], e[0], e[1]))[:5]
top_sql = arus.sort_values(["jumlah", "asal", "tujuan"], ascending=[False, True, True]).head(5)
print("graph:", [(a, b, d["jumlah"]) for a, b, d in top_graf])
print("SQL  :", list(top_sql.itertuples(index=False, name=None)))
print("sama?", [(a, b, d["jumlah"]) for a, b, d in top_graf] == list(top_sql.itertuples(index=False, name=None)))

populer = sorted(G.in_degree(weight="jumlah"), key=lambda kv_: (-kv_[1], kv_[0]))[:5]
print("zona tujuan terpopuler:", populer)

MIN = 50
zona_awal = populer[0][0]

def dua_langkah_graph():
    hasil = set()
    for b, d1 in G[zona_awal].items():
        if d1["jumlah"] < MIN:
            continue
        for c, d2 in G[b].items():
            if d2["jumlah"] >= MIN and c != zona_awal:
                hasil.add(c)
    return hasil

con_d.register("arus", arus)
def dua_langkah_sql():
    return set(con_d.sql(f"""
        SELECT DISTINCT a2.tujuan
        FROM arus a1 JOIN arus a2 ON a1.tujuan = a2.asal
        WHERE a1.asal = {zona_awal} AND a1.jumlah >= {MIN}
          AND a2.jumlah >= {MIN} AND a2.tujuan <> {zona_awal}""").df()["tujuan"])

hg = ukur("2 langkah - graph (NetworkX)", dua_langkah_graph)
hs = ukur("2 langkah - SQL self-join (DuckDB)", dua_langkah_sql)
print("zona awal:", zona_awal, "| terjangkau (graph):", len(hg), "| (SQL):", len(hs), "| sama?", hg == hs)

# Jumlah langkah bebas: cukup ganti k
H = nx.DiGraph([(a, b) for a, b, d in G.edges(data=True) if d["jumlah"] >= MIN])
jangkauan = {}
for k in (1, 2, 3, 4):
    n = len(nx.single_source_shortest_path_length(H, zona_awal, cutoff=k)) - 1
    jangkauan[k] = n
    print(f"dalam <= {k} langkah: {n} zona")
HASIL["jangkauan"] = jangkauan

[hitung arus antar zona] 0.2488 s
node (zona): 263 | edge (arus): 44120
graph: [(46, 205, 789), (146, 129, 787), (74, 221, 765), (221, 166, 765), (46, 221, 761)]
SQL  : [(46, 205, 789), (146, 129, 787), (74, 221, 765), (221, 166, 765), (46, 221, 761)]
sama? True
zona tujuan terpopuler: [(183, 26941), (104, 26916), (146, 26891), (164, 26867), (194, 26847)]
[2 langkah - graph (NetworkX)] 0.0146 s
[2 langkah - SQL self-join (DuckDB)] 0.0106 s
zona awal: 183 | terjangkau (graph): 91 | (SQL): 91 | sama? True
dalam <= 1 langkah: 83 zona
dalam <= 2 langkah: 91 zona
dalam <= 3 langkah: 91 zona
dalam <= 4 langkah: 91 zona


K.6 Indeks Spasial

In [21]:
from shapely import STRtree, box, points

rng = np.random.default_rng(42)
N = 200_000
bujur = rng.normal(-73.97, 0.06, N).clip(-74.25, -73.70)
lintang = rng.normal(40.74, 0.05, N).clip(40.50, 40.92)
titik = points(bujur, lintang)

M = 1000
cx = rng.uniform(-74.05, -73.85, M)
cy = rng.uniform(40.65, 40.85, M)
kotak = [box(x - 0.005, y - 0.005, x + 0.005, y + 0.005) for x, y in zip(cx, cy)]

def cara_loop():
    return [sum(1 for p in titik if k.contains(p)) for k in kotak[:5]]

def cara_numpy():
    hasil = []
    for x, y in zip(cx, cy):
        m = (bujur >= x - 0.005) & (bujur <= x + 0.005) & (lintang >= y - 0.005) & (lintang <= y + 0.005)
        hasil.append(int(m.sum()))
    return hasil

h_loop = ukur("cara 1 - loop Python, 5 kotak", cara_loop)
h_np = ukur("cara 2 - NumPy, 1000 kotak", cara_numpy)
pohon = ukur("bangun indeks STRtree", lambda: STRtree(titik))
h_idx = ukur("cara 3 - STRtree, 1000 kotak", lambda: [len(pohon.query(k, predicate="contains")) for k in kotak])

print("5 kotak pertama - loop:", h_loop, "| STRtree:", h_idx[:5])
print("NumPy == STRtree?", h_np == h_idx, "| loop == STRtree (5 pertama)?", h_loop == h_idx[:5])
print("total titik ditemukan (1000 kotak):", sum(h_idx))
print(f"perkiraan loop untuk 1000 kotak: {T('cara 1 - loop Python, 5 kotak') / 5 * 1000:.0f} s (tidak dijalankan)")

[cara 1 - loop Python, 5 kotak] 18.3994 s
[cara 2 - NumPy, 1000 kotak] 0.5202 s
[bangun indeks STRtree] 0.1202 s
[cara 3 - STRtree, 1000 kotak] 0.3586 s
5 kotak pertama - loop: [1012, 84, 228, 748, 509] | STRtree: [1012, 84, 228, 748, 509]
NumPy == STRtree? True | loop == STRtree (5 pertama)? True
total titik ditemukan (1000 kotak): 417545
perkiraan loop untuk 1000 kotak: 3680 s (tidak dijalankan)


K.7 Delta Lake: versi, time travel, schema enforcement

In [22]:
from deltalake import DeltaTable, write_deltalake
import os
import shutil

KOLOM_DELTA = ["waktu_jemput", "zona_jemput", "zona_tujuan",
               "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"]
data_delta = df[KOLOM_DELTA].sample(n=min(200_000, len(df)), random_state=42).reset_index(drop=True)

PATH_DELTA = "/tmp/trips_delta"
if os.path.exists(PATH_DELTA):
    shutil.rmtree(PATH_DELTA)


write_deltalake(PATH_DELTA, data_delta.iloc[:100_000], mode="overwrite")
dt = DeltaTable(PATH_DELTA)
print("versi:", dt.version(), "| baris:", len(dt.to_pandas()))


write_deltalake(PATH_DELTA, data_delta.iloc[100_000:150_000], mode="append")
dt = DeltaTable(PATH_DELTA)
print("versi:", dt.version(), "| baris:", len(dt.to_pandas()))

sekarang = data_delta.iloc[:150_000]
n_koreksi = int(((sekarang["wilayah_jemput"] == "Bronx") & (sekarang["metode_bayar"] == "Tunai")).sum())
print("baris yang akan dikoreksi:", n_koreksi)

dt.update(
    predicate="wilayah_jemput = 'Bronx' AND metode_bayar = 'Tunai'",
    updates={"total_bayar": "total_bayar + 1.0"}
)

print("versi:", DeltaTable(PATH_DELTA).version())

# TIME TRAVEL
v0 = DeltaTable(PATH_DELTA, version=0).to_pandas()
v1 = DeltaTable(PATH_DELTA, version=1).to_pandas()
terbaru = DeltaTable(PATH_DELTA).to_pandas()

print("baris v0:", len(v0), "| v1:", len(v1), "| terbaru:", len(terbaru))

selisih = terbaru["total_bayar"].sum() - v1["total_bayar"].sum()
print("selisih total_bayar (terbaru - v1):", round(selisih, 2), "| jumlah baris dikoreksi:", n_koreksi)

for h in DeltaTable(PATH_DELTA).history():
    print(h.get("version"), h.get("operation"))

print(sorted(os.listdir(f"{PATH_DELTA}/_delta_log")))

# PENEGAKAN SKEMA: kolom asing harus DITOLAK
tambahan = data_delta.iloc[150_000:150_010].copy()
tambahan["kolom_asing"] = 1

ver_sebelum = DeltaTable(PATH_DELTA).version()

try:
    write_deltalake(PATH_DELTA, tambahan, mode="append")
    print("append kolom asing: BERHASIL (tidak diharapkan)")
    ditolak = False
except Exception as e:
    print("append kolom asing DITOLAK:", type(e).__name__)
    ditolak = True

ver_sesudah = DeltaTable(PATH_DELTA).version()
print("versi setelah percobaan gagal:", ver_sesudah)

HASIL.update(
    n_koreksi=n_koreksi,
    selisih=float(selisih),
    n_v0=len(v0),
    n_v1=len(v1),
    n_terbaru=len(terbaru),
    ditolak=ditolak,
    ver_sebelum=ver_sebelum,
    ver_sesudah=ver_sesudah
)

versi: 0 | baris: 100000
versi: 1 | baris: 150000
baris yang akan dikoreksi: 2517
versi: 2
baris v0: 100000 | v1: 150000 | terbaru: 150000
selisih total_bayar (terbaru - v1): 2517.0 | jumlah baris dikoreksi: 2517
2 UPDATE
1 WRITE
0 WRITE
['00000000000000000000.json', '00000000000000000001.json', '00000000000000000002.json']
append kolom asing DITOLAK: SchemaMismatchError
versi setelah percobaan gagal: 2


K.8 Small file problem pada tabel Delta

In [23]:
dt = DeltaTable(PATH_DELTA)
aktif_awal = len(dt.file_uris())
print("file data aktif sebelum:", aktif_awal, "| versi:", dt.version())

for i in range(20):
    potongan = data_delta.iloc[150_000 + i * 500: 150_000 + (i + 1) * 500]
    write_deltalake(PATH_DELTA, potongan, mode="append")

dt = DeltaTable(PATH_DELTA)
aktif_kecil = len(dt.file_uris())
print("file data aktif setelah 20 append kecil:", aktif_kecil, "| versi:", dt.version(), "| baris:", len(dt.to_pandas()))
t0 = time.perf_counter(); _ = DeltaTable(PATH_DELTA).to_pandas(); t_sebelum = time.perf_counter() - t0

hasil = dt.optimize.compact()
print("hasil compact:", {k: hasil[k] for k in ("numFilesAdded", "numFilesRemoved")})
dt = DeltaTable(PATH_DELTA)
t0 = time.perf_counter(); _ = DeltaTable(PATH_DELTA).to_pandas(); t_sesudah = time.perf_counter() - t0
aktif_compact = len(dt.file_uris())
print("file data aktif setelah compact:", aktif_compact, "| versi:", dt.version(), "| baris:", len(dt.to_pandas()))
print(f"waktu baca sebelum compact: {t_sebelum:.4f} s | sesudah: {t_sesudah:.4f} s")
catatan.append({"langkah": "Delta - baca sebelum compact", "detik": round(t_sebelum, 4)})
catatan.append({"langkah": "Delta - baca sesudah compact", "detik": round(t_sesudah, 4)})

disk = len(glob.glob(f"{PATH_DELTA}/*.parquet"))
print("file parquet di disk:", disk)

kandidat = dt.vacuum(retention_hours=0, enforce_retention_duration=False, dry_run=True)
print("file yang akan dihapus VACUUM (dry run):", len(kandidat))
HASIL.update(aktif_kecil=aktif_kecil, aktif_compact=aktif_compact, disk=disk, n_baris_setelah=len(dt.to_pandas()))

file data aktif sebelum: 1 | versi: 2
file data aktif setelah 20 append kecil: 21 | versi: 22 | baris: 160000
hasil compact: {'numFilesAdded': 1, 'numFilesRemoved': 21}
file data aktif setelah compact: 1 | versi: 23 | baris: 160000
waktu baca sebelum compact: 0.0827 s | sesudah: 0.0519 s
file parquet di disk: 24
file yang akan dihapus VACUUM (dry run): 23


K.9 Merangkum dan Memilih

In [24]:
rekap = pd.DataFrame(catatan)
rekap.to_csv(f"{DIR_SIMPAN}/benchmark_nosql.csv", index=False)
print(rekap.to_string(index=False))
print("\ntersimpan ke:", f"{DIR_SIMPAN}/benchmark_nosql.csv")

                                            langkah   detik
                                  baca trips_bersih  1.2823
                             relasional: muat tabel  3.5589
                             key-value: bangun dict 12.4842
             cari 200 perjalanan - SQL tanpa indeks  5.3148
            cari 200 perjalanan - SQL dengan indeks  0.0039
                    cari 200 perjalanan - key-value  0.0016
                  hitung per wilayah - SQL GROUP BY  0.2166
  hitung per wilayah - key-value (buka semua nilai)  1.3225
DuckDB - hitung per wilayah (langsung dari Parquet)  0.6241
               SQLite - muat seluruh baris ke tabel  4.2900
                        SQLite - hitung per wilayah  0.6314
               DuckDB - hanya wilayah Staten Island  0.0455
                             hitung arus antar zona  0.2488
                       2 langkah - graph (NetworkX)  0.0146
                 2 langkah - SQL self-join (DuckDB)  0.0106
                      cara 1 - loop Pyth

In [25]:
def x(a, b):
    return a / b if b else float("inf")

n_cari = 200
t = {
    "sql_no_idx": T("cari 200 perjalanan - SQL tanpa indeks"),
    "sql_idx":    T("cari 200 perjalanan - SQL dengan indeks"),
    "kv_cari":    T("cari 200 perjalanan - key-value"),
    "sql_agg":    T("hitung per wilayah - SQL GROUP BY"),
    "kv_agg":     T("hitung per wilayah - key-value (buka semua nilai)"),
    "duck":       T("DuckDB - hitung per wilayah (langsung dari Parquet)"),
    "sqlite":     T("SQLite - hitung per wilayah"),
    "duck_pruned": T("DuckDB - hanya wilayah Staten Island"),
    "g2":         T("2 langkah - graph (NetworkX)"),
    "s2":         T("2 langkah - SQL self-join (DuckDB)"),
    "loop5":      T("cara 1 - loop Python, 5 kotak"),
    "numpy":      T("cara 2 - NumPy, 1000 kotak"),
    "bangun":     T("bangun indeks STRtree"),
    "strtree":    T("cara 3 - STRtree, 1000 kotak"),
}
loop1000 = t["loop5"] / 5 * 1000

rekap_pilihan = pd.DataFrame([
    ["Ambil satu data lewat id", "Key-value; tabel berindeks", "Tabel tanpa indeks",
     f"tanpa indeks {t['sql_no_idx']:.4f} s vs berindeks {t['sql_idx']:.4f} s vs key-value {t['kv_cari']:.4f} s (200 pencarian)"],
    ["Menghitung/merangkum banyak baris", "Per kolom (DuckDB pada Parquet)", "Key-value (buka semua nilai)",
     f"DuckDB {t['duck']:.4f} s | SQLite {t['sqlite']:.4f} s | key-value {t['kv_agg']:.4f} s"],
    ["Data dengan field opsional/berubah", "Document", "Tabel (kolom NULL, migrasi)",
     f"{HASIL['n_no_tip']:,} dokumen tanpa tip, {HASIL['n_cuaca']} dokumen field baru tanpa migrasi (kualitatif)"],
    ["Keterhubungan, langkah bebas", "Graph", "SQL (satu JOIN per langkah)",
     f"2 langkah: graph {t['g2']:.4f} s vs SQL {t['s2']:.4f} s; k=1..4: {HASIL['jangkauan']}"],
    ["Mencari lokasi dalam wilayah", "Spasial dengan indeks", "Memeriksa semua titik",
     f"loop ~{loop1000:.0f} s (perkiraan) | NumPy {t['numpy']:.4f} s | STRtree {t['strtree']:.4f} s (+{t['bangun']:.4f} s bangun)"],
    ["Perubahan aman & bisa dibatalkan", "Tabel Delta", "Folder Parquet biasa",
     "versi, time travel, schema enforcement, atomik (kualitatif)"],
], columns=["Kebutuhan", "Cara yang unggul", "Cara yang lemah", "Angka"])
pd.set_option("display.max_colwidth", None)
print(rekap_pilihan.to_string(index=False))
rekap_pilihan.to_csv(f"{DIR_SIMPAN}/rekap_pilihan_k9.csv", index=False)

                         Kebutuhan                Cara yang unggul              Cara yang lemah                                                                             Angka
          Ambil satu data lewat id      Key-value; tabel berindeks           Tabel tanpa indeks tanpa indeks 5.3148 s vs berindeks 0.0039 s vs key-value 0.0016 s (200 pencarian)
 Menghitung/merangkum banyak baris Per kolom (DuckDB pada Parquet) Key-value (buka semua nilai)                            DuckDB 0.6241 s | SQLite 0.6314 s | key-value 1.3225 s
Data dengan field opsional/berubah                        Document  Tabel (kolom NULL, migrasi)          6,021 dokumen tanpa tip, 1 dokumen field baru tanpa migrasi (kualitatif)
      Keterhubungan, langkah bebas                           Graph  SQL (satu JOIN per langkah)   2 langkah: graph 0.0146 s vs SQL 0.0106 s; k=1..4: {1: 83, 2: 91, 3: 91, 4: 91}
      Mencari lokasi dalam wilayah           Spasial dengan indeks        Memeriksa semua titik   loop ~3680 s